<!-- SPDX-FileCopyrightText: Copyright (c) 2026 NVIDIA CORPORATION & AFFILIATES. All rights reserved.
SPDX-License-Identifier: OpenMDW-1.1 -->

# Quantize Cosmos3-Super Image-to-Video to FP8

Cosmos3-Super Image-to-Video generates a video from a single conditioning image. Like the
Text-to-Image model, it ships as a full-quality model and a **4-step distilled** version.
This notebook quantizes both to FP8 — the base model first, then the distilled model.

Image-to-Video calibration conditions on a clean first frame (a real image, encoded by the
VAE) and holds it fixed while denoising the rest of the clip — matching how the model is
served. You provide the conditioning images in the next step.

## 1. Prerequisites

Use a Linux machine with an NVIDIA GPU, model access on Hugging Face, and either
`uvx hf@latest auth login` or `HF_TOKEN` set. Quantizing the Super (32B) checkpoints
needs a GPU with enough memory to hold the model in bf16 plus the quantizers; Nano (8B)
is comfortable on a single 48 GB GPU.

You also need free disk for the Hugging Face checkpoint cache and for the FP8 output
(each output is roughly the size of the bf16 source). Point `HF_HOME` at a large volume
in the next step.

> **Headless servers:** if you see `libGL.so.1: cannot open shared object file` when the
> VAE loads, install the system graphics libraries:
>
> ```bash
> apt-get install -y libgl1 libglib2.0-0
> ```

## 2. Configure Paths and Environment

The defaults are relative to this `cosmos` checkout and use the CUDA 13 (or 12.8) Torch
backend depending on your system. Override any of these before running the cell:

```bash
export COSMOS3_QUANTIZE_VENV=/path/to/.venv-cosmos3-quantize
export COSMOS3_TORCH_BACKEND=cu130       # or cu128
export HF_HOME=/path/to/large/huggingface/cache
export OUTPUT_ROOT=/path/to/fp8/outputs
```

In [1]:
from pathlib import Path
import os


def find_repo_root(start: Path) -> Path:
    for path in [start, *start.parents]:
        if (path / 'README.md').exists() and (path / 'cookbooks').exists():
            return path
    return start


def configure_quantize_environment() -> None:
    global COSMOS_ROOT, QUANTIZE_ROOT, COSMOS3_QUANTIZE_VENV
    global COSMOS3_TORCH_BACKEND, OUTPUT_ROOT

    COSMOS_ROOT = find_repo_root(Path.cwd().resolve())
    QUANTIZE_ROOT = COSMOS_ROOT / 'cookbooks' / 'cosmos3' / 'quantization'
    COSMOS3_QUANTIZE_VENV = Path(
        os.environ.get('COSMOS3_QUANTIZE_VENV', COSMOS_ROOT / '.venv-cosmos3-quantize')
    ).resolve()
    COSMOS3_TORCH_BACKEND = os.environ.get('COSMOS3_TORCH_BACKEND', 'cu130')
    OUTPUT_ROOT = Path(os.environ.get('OUTPUT_ROOT', QUANTIZE_ROOT / 'outputs')).resolve()

    os.environ['COSMOS3_QUANTIZE_VENV'] = str(COSMOS3_QUANTIZE_VENV)
    os.environ['COSMOS3_TORCH_BACKEND'] = COSMOS3_TORCH_BACKEND
    os.environ['OUTPUT_ROOT'] = str(OUTPUT_ROOT)
    os.environ.setdefault('UV_LINK_MODE', 'copy')
    os.environ.setdefault('HF_HOME', str(Path.home() / '.cache' / 'huggingface'))
    # Keep the datasets cache writable and separate from the shared hub lock dir.
    os.environ.setdefault('HF_DATASETS_CACHE', str(Path(os.environ['HF_HOME']) / 'datasets'))
    OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

configure_quantize_environment()
print('COSMOS_ROOT   :', COSMOS_ROOT)
print('QUANTIZE_ROOT :', QUANTIZE_ROOT)
print('OUTPUT_ROOT   :', OUTPUT_ROOT)
print('venv          :', COSMOS3_QUANTIZE_VENV)

COSMOS_ROOT   : <COSMOS>
QUANTIZE_ROOT : <COSMOS>/cookbooks/cosmos3/quantization
OUTPUT_ROOT   : <COSMOS>/cookbooks/cosmos3/quantization/outputs
venv          : <COSMOS>/cookbooks/cosmos3/quantization/.venv-cosmos3-quantize


## 3. Install Dependencies

This creates a dedicated virtual environment with PyTorch, NVIDIA TensorRT Model
Optimizer (ModelOpt — the FP8 quantization engine), Diffusers (VAE + schedulers) and
Transformers (tokenizer), then registers a Jupyter kernel for it. Run this once.

In [2]:
%%bash
set -euo pipefail

if ! command -v uv >/dev/null 2>&1; then
  echo 'uv is not installed. Install it first: https://docs.astral.sh/uv/getting-started/installation/'
  exit 1
fi

export UV_LINK_MODE="${UV_LINK_MODE:-copy}"
uv venv "$COSMOS3_QUANTIZE_VENV" --python 3.13 --seed --managed-python --allow-existing
source "$COSMOS3_QUANTIZE_VENV/bin/activate"
uv pip install --torch-backend="$COSMOS3_TORCH_BACKEND" \
  "diffusers @ git+https://github.com/huggingface/diffusers.git" \
  "nvidia-modelopt[torch]" \
  accelerate datasets huggingface_hub imageio imageio-ffmpeg ipykernel \
  numpy pillow safetensors torch torchvision transformers

"$COSMOS3_QUANTIZE_VENV/bin/python" -m ipykernel install --user \
  --name cosmos3-quantize \
  --display-name "Cosmos3 Quantize (Python 3.13)"

echo
echo "Installed into: $COSMOS3_QUANTIZE_VENV"
echo "Next: switch this notebook kernel to: Cosmos3 Quantize (Python 3.13)"

Using CPython 

3.13.13
Creating virtual environment with seed packages at: <HOME>

kutak_other<COSMOS>/cookbooks/cosmos3/quantization/.venv-cosmos3-quantize


Using Python 3.13.13 environment at: <COSMOS>/cookbooks/cosmos3/qua

ntize/.venv-cosmos3-quantize


...

Checked 116 packages in 10ms

age `nvidia-modelopt==0.45.0` does not have an extra named `torch`


Installed kernelspec cosmos3-quantize in <HOME>/.local/share/jupyter/kernels/cosmos3-quantize



Installed into: <COSMOS>/cookbooks/cosmos3/quantization/.venv-cosmos3-quan

tize
Next: switch this notebook kernel to: Cosmos3 Quantize (Python 3.13)


## 4. Select the Quantization Kernel

The install cell registers the `Cosmos3 Quantize (Python 3.13)` Jupyter kernel.

**Switch this notebook to that kernel**, then run the restore cell below before
continuing. It can take a moment for a new kernel to appear in the notebook interface.

In [3]:
# Run this cell immediately after switching to the Cosmos3 Quantize kernel.
# It restores the same paths and cache settings as the Configure cell above.
from pathlib import Path
import os


def find_repo_root(start: Path) -> Path:
    for path in [start, *start.parents]:
        if (path / 'README.md').exists() and (path / 'cookbooks').exists():
            return path
    return start


def configure_quantize_environment() -> None:
    global COSMOS_ROOT, QUANTIZE_ROOT, COSMOS3_QUANTIZE_VENV
    global COSMOS3_TORCH_BACKEND, OUTPUT_ROOT

    COSMOS_ROOT = find_repo_root(Path.cwd().resolve())
    QUANTIZE_ROOT = COSMOS_ROOT / 'cookbooks' / 'cosmos3' / 'quantization'
    COSMOS3_QUANTIZE_VENV = Path(
        os.environ.get('COSMOS3_QUANTIZE_VENV', COSMOS_ROOT / '.venv-cosmos3-quantize')
    ).resolve()
    COSMOS3_TORCH_BACKEND = os.environ.get('COSMOS3_TORCH_BACKEND', 'cu130')
    OUTPUT_ROOT = Path(os.environ.get('OUTPUT_ROOT', QUANTIZE_ROOT / 'outputs')).resolve()

    os.environ['COSMOS3_QUANTIZE_VENV'] = str(COSMOS3_QUANTIZE_VENV)
    os.environ['COSMOS3_TORCH_BACKEND'] = COSMOS3_TORCH_BACKEND
    os.environ['OUTPUT_ROOT'] = str(OUTPUT_ROOT)
    os.environ.setdefault('UV_LINK_MODE', 'copy')
    os.environ.setdefault('HF_HOME', str(Path.home() / '.cache' / 'huggingface'))
    # Keep the datasets cache writable and separate from the shared hub lock dir.
    os.environ.setdefault('HF_DATASETS_CACHE', str(Path(os.environ['HF_HOME']) / 'datasets'))
    OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

configure_quantize_environment()
print('environment restored — OUTPUT_ROOT:', OUTPUT_ROOT)

environment restored — OUTPUT_ROOT: <COSMOS>/cookbooks/cosmos3/quantization/outputs


## 5. Verify GPU and Python Environment

Confirm the kernel sees a GPU and the quantization engine imports cleanly.

In [4]:
import torch, modelopt
print('torch          :', torch.__version__)
print('CUDA available :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU            :', torch.cuda.get_device_name(0))
print('ModelOpt       :', modelopt.__version__)

torch          : 2.13.0+cu130
CUDA available : True
GPU            : NVIDIA B200
ModelOpt       : 0.45.0


## 6. Load the Quantization Toolkit

The cookbook ships the Cosmos3 model and the complete FP8 recipe in `quantization/src/`:
loading, calibration, and export. `quantize_fp8_checkpoint(...)` runs the whole
pipeline for one checkpoint — load the model, **calibrate** it by replaying real
denoising so the quantizer sees representative activations, quantize to FP8, and write a
drop-in checkpoint you can serve.

The two helpers below wrap that call (resolving the Hugging Face checkpoint) and print a
short summary of what was produced.

In [5]:
import sys
sys.path.insert(0, str(QUANTIZE_ROOT))

from huggingface_hub import snapshot_download
from safetensors import safe_open
import json
import src
from src import (quantize_fp8_checkpoint,
                 SHAPE_VIDEO, SHAPE_IMAGE, SHAPE_VIDEO_DEMO, SHAPE_IMAGE_DEMO,
                 SAMPLER_VIDEO_BASE, SAMPLER_IMAGE_BASE, SAMPLER_DISTILLED)

# DEMO=1 (default): one calibration prompt at a small shape, so a run takes minutes.
# Set DEMO=0 for the production shape and 8 calibration prompts (the shipped recipe).
DEMO = os.environ.get('DEMO', '1') == '1'
NUM_SAMPLES = 1 if DEMO else 8
print(f'DEMO={DEMO}  (NUM_SAMPLES={NUM_SAMPLES})')


def resolve_checkpoint(hf_repo, override_env):
    """Local dir from `override_env` if set, else the Hugging Face snapshot."""
    override = os.environ.get(override_env)
    return Path(override) if override else Path(snapshot_download(hf_repo))


def inspect_checkpoint(output_dir):
    """Print a short, human-readable summary of an FP8 checkpoint."""
    output_dir = Path(output_dir)
    tdir = output_dir / 'transformer'
    n_fp8 = n_scale = 0
    example = []
    for shard in sorted(tdir.glob('*.safetensors')):
        with safe_open(str(shard), framework='pt') as h:
            for k in h.keys():
                if k.endswith(('.input_scale', '.weight_scale')):
                    n_scale += 1
                    if k.endswith('.input_scale') and len(example) < 3:
                        example.append((k, float(h.get_tensor(k).reshape(-1)[0])))
                elif k.endswith('.weight') and h.get_slice(k).get_dtype() == 'F8_E4M3':
                    n_fp8 += 1
    qcfg = json.load(open(output_dir / 'hf_quant_config.json'))
    print(f'FP8 checkpoint: {output_dir}')
    print(f"  quant_algo   : {qcfg.get('quant_algo')}  (method={qcfg.get('quant_method')})")
    print(f'  FP8 weights  : {n_fp8}')
    print(f'  scale tensors: {n_scale}')
    for k, v in example:
        print(f'    e.g. {k} = {v:.6g}')

<COSMOS>/cookbooks/cosmos3/quantization/.venv-cosmos3-quantize/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


DEMO=True  (NUM_SAMPLES=1)


<COSMOS>/cookbooks/cosmos3/quantization/.venv-cosmos3-quantize/lib/python3.13/site-packages/modelopt/torch/__init__.py:51: UserWarning: transformers 5.14.1 is not tested with current version of modelopt and may cause issues. Please install recommended version with `pip install -U nvidia-modelopt[hf]` if working with HF models.
  _warnings.warn(


## Conditioning Images

Point `I2V_COND_DIR` at a local folder of images to condition on. If you leave it unset,
the recipe falls back to a public calibration image dataset (`nemotron_vlm_dataset_v2`),
which requires Hugging Face access to that dataset.

In [6]:
I2V_COND_DIR = os.environ.get('I2V_COND_DIR')  # e.g. '/data/my_cond_images'
I2V_COND_DATASET = None if I2V_COND_DIR else 'nemotron_vlm_dataset_v2'
print('conditioning from:', I2V_COND_DIR or f'dataset {I2V_COND_DATASET}')

conditioning from: dataset nemotron_vlm_dataset_v2


## Cosmos3-Super Image-to-Video (Base)

The full-quality image-to-video model, served with a 50-step UniPC sampler.

### Quantize

In [7]:
input_dir = resolve_checkpoint('nvidia/Cosmos3-Super-Image2Video', 'C3_I2V_DIR')
output_dir = OUTPUT_ROOT / 'super-i2v-fp8'

quantize_fp8_checkpoint(
    input_dir=input_dir, output_dir=output_dir,
    profile='i2v', sampler=SAMPLER_VIDEO_BASE, shape=SHAPE_VIDEO_DEMO if DEMO else SHAPE_VIDEO,
    num_samples=NUM_SAMPLES,
    i2v_cond_dir=I2V_COND_DIR, i2v_cond_dataset=I2V_COND_DATASET,
)


Fetching 69 files:   0%|                                                                             | 0/69 [00:00<?, ?it/s]


Fetching 69 files:   1%|█                                                                    | 1/69 [00:00<00:28,  2.40it/s]


Fetching 69 files:  10%|███████                                                              | 7/69 [00:00<00:05, 10.50it/s]


Fetching 69 files:  13%|█████████                                                            | 9/69 [00:00<00:05, 11.15it/s]


Fetching 69 files:  16%|██████████▊                                                         | 11/69 [00:01<00:04, 11.81it/s]

...

[export] overlaid 896 fp8 weights + 1792 scales onto bf16 base (dropped 4 vae2llm/llm2vae); 3212 tensors total


[export] wrote 3212 tensors across 14 shard(s)
[export] wrote 3212 tensors (fp8) to <COSMOS>/cookbooks/cosmos3/quantization/outputs/.quantized_transformer_i2v.tmp


[assemble] moving quantized -> <COSMOS>/cookbooks/cosmos3/quantization/outputs/super-i2v-fp8/transformer
[assemble] linking assets -> <HF_HOME>/models--nvidia--Cosmos3-Super-Image2Video/snapshots/100d2f5ba573e75e4bc8f159388e8f8353312dc9/assets
[assemble] linking BIAS.md -> <HF_HOME>/models--nvidia--Cosmos3-Super-Image2Video/snapshots/100d2f5ba573e75e4bc8f159388e8f8353312dc9/BIAS.md
[assemble] linking PRIVACY.md -> <HF_HOME>/models--nvidia--Cosmos3-Super-Image2Video/snapshots/100d2f5ba573e75e4bc8f159388e8f8353312dc9/PRIVACY.md
[assemble] linking SAFETY.md -> <HF_HOME>/models--nvidia--Cosmos3-Super-Image2Video/snapshots/100d2f5ba573e75e4bc8f159388e8f8353312dc9/SAFETY.md
[assemble] linking .gitattributes -> <HF_HOME>/models--nvidia--Cosmos3-Super-Image2Video/snapshots/100d2f5ba573e75e4bc8f159388e8f8353312dc9/.gitattributes
[assemble] linking EXPLAINABILITY.md -> <HF_HOME>/models--nvidia--Cosmos3-Super-Image2Video/snapshots/100d2f5ba573e75e4bc8f159388e8f8353312dc9/EXPLAINABILITY.md
[assemb

[assemble] wrote model.safetensors.index.json (3563 tensors, 66775154528 bytes)
[assemble] wrote <COSMOS>/cookbooks/cosmos3/quantization/outputs/super-i2v-fp8/hf_quant_config.json for upstream-vLLM discovery
[assemble] drop-in dir ready at <COSMOS>/cookbooks/cosmos3/quantization/outputs/super-i2v-fp8
[done] <COSMOS>/cookbooks/cosmos3/quantization/outputs/super-i2v-fp8


PosixPath('<COSMOS>/cookbooks/cosmos3/quantization/outputs/super-i2v-fp8')

### Inspect the FP8 Checkpoint

A quick look at the drop-in checkpoint just written: the quantization method, how many
weights were converted to FP8, and the per-tensor scales that accompany them.

In [8]:
inspect_checkpoint(OUTPUT_ROOT / 'super-i2v-fp8')

FP8 checkpoint: <COSMOS>/cookbooks/cosmos3/quantization/outputs/super-i2v-fp8
  quant_algo   : FP8  (method=modelopt)
  FP8 weights  : 896
  scale tensors: 1792
    e.g. layers.0.mlp.down_proj.input_scale = 0.0262277
    e.g. layers.0.mlp.gate_proj.input_scale = 0.00244141
    e.g. layers.0.mlp.up_proj.input_scale = 0.00244141


## Cosmos3-Super Image-to-Video — 4-Step Distilled

The distilled model is the **same network**, served in just 4 steps. As with Text-to-Image,
the only change for quantization is the **sampler**; the image conditioning is identical:

| | scheduler | steps | guidance |
|---|---|---|---|
| base | UniPC | 50 | 6.0 |
| distilled | FlowMatchEuler (4 fixed steps) | 4 | 1.0 (guidance off) |

### Quantize

In [9]:
input_dir = resolve_checkpoint('nvidia/Cosmos3-Super-Image2Video-4Step', 'C3_I2V_4STEP_DIR')
output_dir = OUTPUT_ROOT / 'super-i2v-distilled-fp8'

quantize_fp8_checkpoint(
    input_dir=input_dir, output_dir=output_dir,
    profile='i2v', sampler=SAMPLER_DISTILLED, shape=SHAPE_VIDEO_DEMO if DEMO else SHAPE_VIDEO,
    num_samples=NUM_SAMPLES,
    i2v_cond_dir=I2V_COND_DIR, i2v_cond_dataset=I2V_COND_DATASET,
)


Fetching 66 files:   0%|                                                                             | 0/66 [00:00<?, ?it/s]


Fetching 66 files:   2%|█                                                                    | 1/66 [00:00<00:22,  2.86it/s]


Fetching 66 files:  11%|███████▎                                                             | 7/66 [00:00<00:04, 12.57it/s]


Fetching 66 files:  18%|████████████▎                                                       | 12/66 [00:00<00:02, 19.96it/s]


Fetching 66 files:  23%|███████████████▍                                                    | 15/66 [00:01<00:03, 13.97it/s]

...

[export] overlaid 896 fp8 weights + 1792 scales onto bf16 base (dropped 4 vae2llm/llm2vae); 3212 tensors total


[export] wrote 3212 tensors across 14 shard(s)
[export] wrote 3212 tensors (fp8) to <COSMOS>/cookbooks/cosmos3/quantization/outputs/.quantized_transformer_i2v.tmp


[assemble] moving quantized -> <COSMOS>/cookbooks/cosmos3/quantization/outputs/super-i2v-distilled-fp8/transformer
[assemble] linking assets -> <HF_HOME>/models--nvidia--Cosmos3-Super-Image2Video-4Step/snapshots/fda1636fde300b845e6d40f9e510876f17f6c740/assets
[assemble] linking README.md -> <HF_HOME>/models--nvidia--Cosmos3-Super-Image2Video-4Step/snapshots/fda1636fde300b845e6d40f9e510876f17f6c740/README.md
[assemble] linking BIAS.md -> <HF_HOME>/models--nvidia--Cosmos3-Super-Image2Video-4Step/snapshots/fda1636fde300b845e6d40f9e510876f17f6c740/BIAS.md
[assemble] linking EXPLAINABILITY.md -> <HF_HOME>/models--nvidia--Cosmos3-Super-Image2Video-4Step/snapshots/fda1636fde300b845e6d40f9e510876f17f6c740/EXPLAINABILITY.md
[assemble] linking .gitattributes -> <HF_HOME>/models--nvidia--Cosmos3-Super-Image2Video-4Step/snapshots/fda1636fde300b845e6d40f9e510876f17f6c740/.gitattributes
[assemble] linking PRIVACY.md -> <HF_HOME>/models--nvidia--Cosmos3-Super-Image2Video-4Step/snapshots/fda1636fde300

[assemble] wrote model.safetensors.index.json (3212 tensors, 65584620928 bytes)
[assemble] wrote <COSMOS>/cookbooks/cosmos3/quantization/outputs/super-i2v-distilled-fp8/hf_quant_config.json for upstream-vLLM discovery
[assemble] drop-in dir ready at <COSMOS>/cookbooks/cosmos3/quantization/outputs/super-i2v-distilled-fp8
[done] <COSMOS>/cookbooks/cosmos3/quantization/outputs/super-i2v-distilled-fp8


PosixPath('<COSMOS>/cookbooks/cosmos3/quantization/outputs/super-i2v-distilled-fp8')

### Inspect the FP8 Checkpoint

A quick look at the drop-in checkpoint just written: the quantization method, how many
weights were converted to FP8, and the per-tensor scales that accompany them.

In [10]:
inspect_checkpoint(OUTPUT_ROOT / 'super-i2v-distilled-fp8')

FP8 checkpoint: <COSMOS>/cookbooks/cosmos3/quantization/outputs/super-i2v-distilled-fp8
  quant_algo   : FP8  (method=modelopt)
  FP8 weights  : 896
  scale tensors: 1792
    e.g. layers.0.mlp.down_proj.input_scale = 0.0262277
    e.g. layers.0.mlp.gate_proj.input_scale = 0.00244141
    e.g. layers.0.mlp.up_proj.input_scale = 0.00244141
